# 訓練 Freminet 語音模型（GPT-SoVITS v2ProPlus）

用 notebook 代替 WebUI 執行 1A → 1B → 匯出，參數跟 README 的方案一樣。

- 訓練腳本用 GPT-SoVITS 整合包裏的 `runtime\python.exe` 執行，這個 notebook 本身只用 Python 標準庫，任何 Python 3 kernel 都可以
- 訓練前：筆電接電源、電源模式設成「最佳效能」、關掉佔用顯卡的程式
- **不要同時開著 WebUI 訓練**，會搶顯存
- **可以直接「全部執行」**：會用最後一輪的模型和預設的參考音頻，匯出到 `Models/Freminet/`
- 想換別的輪數或參考音頻：先用 WebUI 的 1C 推理試聽，在第 8 部分填好，再單獨執行那一格

## 1. 設定

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

PROJECT = Path(r"E:\genshin-project")
GSV = Path(r"D:\characters\GPT-SoVITS-Rin\GPT-SoVITS-v2pro-20250604")
PYTHON = GSV / "runtime" / "python.exe"

VERSION = "v2ProPlus"  # 爆顯存且 batch size 1 也不行時改成 "v2Pro"
EXP_NAME = "Freminet"
INP_TEXT = PROJECT / "freminet_chinese" / "freminet.list"
MODELS_DIR = PROJECT / "Models"

GPU = "0"
IS_HALF = True

# SoVITS
SOVITS_BATCH = 2  # 順利可試 3–4，爆顯存改 1
SOVITS_EPOCHS = 8
SOVITS_SAVE_EVERY = 4
TEXT_LOW_LR_RATE = 0.4

# GPT
GPT_BATCH = 2  # 順利可試 3–4，爆顯存改 1
GPT_EPOCHS = 15  # 10–15，太多會過擬合
GPT_SAVE_EVERY = 5

EXP_DIR = GSV / "logs" / EXP_NAME
TEMP_DIR = GSV / "TEMP"
SOVITS_WEIGHTS = GSV / f"SoVITS_weights_{VERSION}"
GPT_WEIGHTS = GSV / f"GPT_weights_{VERSION}"

PRETRAINED = GSV / "GPT_SoVITS" / "pretrained_models"
PRETRAINED_S2G = PRETRAINED / "v2Pro" / f"s2G{VERSION}.pth"
PRETRAINED_S2D = PRETRAINED / "v2Pro" / f"s2D{VERSION}.pth"
PRETRAINED_S1 = PRETRAINED / "s1v3.ckpt"
BERT_DIR = PRETRAINED / "chinese-roberta-wwm-ext-large"
HUBERT_DIR = PRETRAINED / "chinese-hubert-base"
SV_PATH = PRETRAINED / "sv" / "pretrained_eres2netv2w24s4ep4.ckpt"

S2_CONFIG = GSV / "GPT_SoVITS" / "configs" / f"s2{VERSION}.json"
S1_CONFIG = GSV / "GPT_SoVITS" / "configs" / "s1longer-v2.yaml"

## 2. 檢查

In [ ]:
for p in [PYTHON, INP_TEXT, PRETRAINED_S2G, PRETRAINED_S2D, PRETRAINED_S1,
          BERT_DIR, HUBERT_DIR, SV_PATH, S2_CONFIG, S1_CONFIG]:
    assert p.exists(), f"找不到：{p}"

rows = [l for l in INP_TEXT.read_text(encoding="utf-8").splitlines() if l.strip()]
missing = [r.split("|")[0] for r in rows if not Path(r.split("|")[0]).exists()]
assert not missing, f"有 {len(missing)} 個音頻找不到，例如 {missing[:3]}"
print(f"訓練列表：{len(rows)} 段，音頻都在")

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"])

執行訓練腳本的工具函數（相當於 WebUI 在背後做的事）。

In [ ]:
BASE_ENV = {
    "version": VERSION,
    "is_half": str(IS_HALF),
    "exp_name": EXP_NAME,
    "opt_dir": str(EXP_DIR),
    "inp_text": str(INP_TEXT),
    "inp_wav_dir": "",  # 列表裏已經是完整路徑
    "i_part": "0",
    "all_parts": "1",
    "_CUDA_VISIBLE_DEVICES": GPU,
    "PYTHONIOENCODING": "utf-8",
}


def run(*args, env=None):
    e = os.environ.copy()
    e.update(BASE_ENV)
    e.update({k: str(v) for k, v in (env or {}).items()})
    e["PATH"] = str(GSV / "runtime") + os.pathsep + e["PATH"]
    cmd = [str(PYTHON), "-s", *map(str, args)]
    print(">", " ".join(cmd))
    p = subprocess.Popen(cmd, cwd=GSV, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"執行失敗（exit code {p.returncode}）：{args[0]}")


def load_yaml(path):
    # notebook 的 Python 不一定有 PyYAML，借用整合包的 Python 讀
    out = subprocess.run([str(PYTHON), "-c",
                          "import json,sys,yaml;print(json.dumps(yaml.safe_load(open(sys.argv[1],encoding='utf-8'))))",
                          str(path)], capture_output=True, text=True, check=True)
    return json.loads(out.stdout)

## 3. 1A 資料格式化（一鍵三連）

文本 → BERT 特徵、音頻 → HuBERT 特徵和 32k 音頻、說話人特徵（v2Pro 系列才需要）、語義 token。
已經做過的步驟會跳過；要重做就先刪掉 `logs/Freminet` 資料夾。

In [ ]:
EXP_DIR.mkdir(parents=True, exist_ok=True)

# 1Aa 文本
text_path = EXP_DIR / "2-name2text.txt"
if text_path.exists() and len(text_path.read_text(encoding="utf-8").strip("\n").split("\n")) >= 2:
    print("1Aa 已完成，跳過")
else:
    run("GPT_SoVITS/prepare_datasets/1-get-text.py", env={"bert_pretrained_dir": BERT_DIR})
    part = EXP_DIR / "2-name2text-0.txt"
    text = part.read_text(encoding="utf-8").strip("\n")
    assert text, "1Aa 失敗：沒有輸出"
    text_path.write_text(text + "\n", encoding="utf-8")
    part.unlink()

# 1Ab 音頻特徵 + 說話人特徵
run("GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py", env={"cnhubert_base_dir": HUBERT_DIR})
if "Pro" in VERSION:
    run("GPT_SoVITS/prepare_datasets/2-get-sv.py", env={"sv_path": SV_PATH})

# 1Ac 語義 token
semantic_path = EXP_DIR / "6-name2semantic.tsv"
if semantic_path.exists() and semantic_path.stat().st_size >= 31:
    print("1Ac 已完成，跳過")
else:
    run("GPT_SoVITS/prepare_datasets/3-get-semantic.py",
        env={"pretrained_s2G": PRETRAINED_S2G, "s2config_path": S2_CONFIG})
    part = EXP_DIR / "6-name2semantic-0.tsv"
    semantic = part.read_text(encoding="utf-8").strip("\n")
    semantic_path.write_text("item_name\tsemantic_audio\n" + semantic + "\n", encoding="utf-8")
    part.unlink()

n_text = len(text_path.read_text(encoding="utf-8").strip("\n").split("\n"))
n_sem = len(semantic_path.read_text(encoding="utf-8").strip("\n").split("\n")) - 1
print(f"\n1A 完成：文本 {n_text} 段，語義 {n_sem} 段（列表 {len(rows)} 段）")

## 4. 1Ba 訓練 SoVITS

爆顯存（OOM）時：① `SOVITS_BATCH = 1` → ② 去掉超過 30 秒的長音頻 → ③ 改用 v2Pro。

In [ ]:
data = json.loads(S2_CONFIG.read_text(encoding="utf-8"))
if not IS_HALF:
    data["train"]["fp16_run"] = False
data["train"].update({
    "batch_size": SOVITS_BATCH if IS_HALF else max(1, SOVITS_BATCH // 2),
    "epochs": SOVITS_EPOCHS,
    "text_low_lr_rate": TEXT_LOW_LR_RATE,
    "pretrained_s2G": str(PRETRAINED_S2G),
    "pretrained_s2D": str(PRETRAINED_S2D),
    "if_save_latest": True,
    "if_save_every_weights": True,
    "save_every_epoch": SOVITS_SAVE_EVERY,
    "gpu_numbers": GPU,
    "grad_ckpt": False,
    "lora_rank": "32",
})
data["model"]["version"] = VERSION
data["data"]["exp_dir"] = data["s2_ckpt_dir"] = str(EXP_DIR)
data["save_weight_dir"] = str(SOVITS_WEIGHTS)
data["name"] = EXP_NAME
data["version"] = VERSION

(EXP_DIR / f"logs_s2_{VERSION}").mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)
s2_tmp = TEMP_DIR / "tmp_s2.json"
s2_tmp.write_text(json.dumps(data), encoding="utf-8")

run("GPT_SoVITS/s2_train.py", "--config", s2_tmp)

## 5. 1Bb 訓練 GPT

DPO 關閉（6GB 不夠）。聲音亂念、重複、吞字時，換用輪數比較少的 GPT 模型。

In [ ]:
data = load_yaml(S1_CONFIG)
if not IS_HALF:
    data["train"]["precision"] = "32"
data["train"].update({
    "batch_size": GPT_BATCH if IS_HALF else max(1, GPT_BATCH // 2),
    "epochs": GPT_EPOCHS,
    "save_every_n_epoch": GPT_SAVE_EVERY,
    "if_save_every_weights": True,
    "if_save_latest": True,
    "if_dpo": False,
    "half_weights_save_dir": str(GPT_WEIGHTS),
    "exp_name": EXP_NAME,
})
data["pretrained_s1"] = str(PRETRAINED_S1)
data["train_semantic_path"] = str(EXP_DIR / "6-name2semantic.tsv")
data["train_phoneme_path"] = str(EXP_DIR / "2-name2text.txt")
data["output_dir"] = str(EXP_DIR / f"logs_s1_{VERSION}")

(EXP_DIR / "logs_s1").mkdir(parents=True, exist_ok=True)
s1_tmp = TEMP_DIR / "tmp_s1.yaml"
s1_tmp.write_text(json.dumps(data), encoding="utf-8")  # JSON 也是合法的 YAML

run("GPT_SoVITS/s1_train.py", "--config_file", s1_tmp, env={"hz": "25hz"})

## 6. 訓練結果

In [ ]:
def list_weights():
    sovits = sorted(SOVITS_WEIGHTS.glob(f"{EXP_NAME}_e*.pth"), key=lambda p: p.stat().st_mtime)
    gpt = sorted(GPT_WEIGHTS.glob(f"{EXP_NAME}-e*.ckpt"), key=lambda p: p.stat().st_mtime)
    return sovits, gpt

sovits_list, gpt_list = list_weights()
print("SoVITS：")
for p in sovits_list:
    print(f"  {p.name}  ({p.stat().st_size / 2**20:.0f} MB)")
print("GPT：")
for p in gpt_list:
    print(f"  {p.name}  ({p.stat().st_size / 2**20:.0f} MB)")

## 7. 推理測試（WebUI，可選）

全部執行時會跳過這一步，直接用預設值匯出。之後想挑更好的組合：

1. 執行 `go-webui.bat` → 1C 推理 → 開啟 TTS 推理 WebUI
2. 參考音頻：挑一段 3–10 秒、語氣平穩、沒有喘氣或叫聲的原聲，文本從 `metadata.csv` 查
3. 試不同輪數的 SoVITS / GPT 組合，挑效果最好的，填到下面再單獨執行匯出那一格（會覆蓋之前匯出的）

| 問題 | 解決方法 |
|---|---|
| 聲音不像 | 換參考音頻，或 SoVITS 多訓練幾輪 |
| 亂念、重複、吞字 | 換用輪數比較少的 GPT 模型 |
| 雜音、電音 | 換其他輪數的 SoVITS 模型，或換參考音頻 |

## 8. 匯出到 `Models/`

模型檔名留空 = 用最後一輪的；參考音頻有預設值。會匯出到 `Models/Freminet/`：

```
Models/Freminet/
├── Freminet_e8_s....pth     SoVITS 模型
├── Freminet-e15.ckpt        GPT 模型
├── ref.wav                  參考音頻
└── model.json               版本、檔名、參考文本（之後聊天程式讀這個）
```

In [ ]:
SOVITS_PICK = ""  # 例如 "Freminet_e8_s1416.pth"，留空用最新的
GPT_PICK = ""     # 例如 "Freminet-e15.ckpt"，留空用最新的
# 參考音頻（在 freminet_chinese/ 裏），預設：6.6 秒，「请注意安全，不要去陌生的水域。遇见危险的话，也请尽快上浮。」
REF_AUDIO = "7ee9712a0f7a3869.wav"

sovits_list, gpt_list = list_weights()
assert sovits_list and gpt_list, "還沒有訓練好的模型"
sovits = SOVITS_WEIGHTS / SOVITS_PICK if SOVITS_PICK else sovits_list[-1]
gpt = GPT_WEIGHTS / GPT_PICK if GPT_PICK else gpt_list[-1]
assert sovits.exists(), f"找不到：{sovits}"
assert gpt.exists(), f"找不到：{gpt}"

out = MODELS_DIR / EXP_NAME
out.mkdir(parents=True, exist_ok=True)
for old in [*out.glob("*.pth"), *out.glob("*.ckpt")]:
    old.unlink()
shutil.copy2(sovits, out / sovits.name)
shutil.copy2(gpt, out / gpt.name)

info = {"version": VERSION, "sovits": sovits.name, "gpt": gpt.name,
        "ref_audio": None, "ref_text": None, "ref_lang": "zh"}

if REF_AUDIO:
    # 從訓練列表查文本（已經清理過）
    texts = {Path(r.split("|")[0]).name: r.split("|")[3] for r in rows}
    assert REF_AUDIO in texts, f"{REF_AUDIO} 不在 freminet.list 裏"
    shutil.copy2(PROJECT / "freminet_chinese" / REF_AUDIO, out / "ref.wav")
    info["ref_audio"] = "ref.wav"
    info["ref_text"] = texts[REF_AUDIO]
else:
    print("⚠ 沒有指定參考音頻，只匯出模型")

(out / "model.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"已匯出到 {out}")
for p in sorted(out.iterdir()):
    print(f"  {p.name}")
print(json.dumps(info, ensure_ascii=False, indent=2))